<div style="background-color: white; padding: 10px;">
<center>
    <img style="padding-left:15px"  height='50px' src="https://www.norkart.no/hubfs/norkart-logo-default.svg">
    <img style="padding-left:15px"  height='50px' src="https://www.kartverket.no/public/images/logo/kartverket-logo-large2.svg">
    <br />
    <img style="padding-right:15px" height='50px' src="https://kartai.no/wp-content/uploads/2025/03/cropped-KartAi-med-partnere-2048x1145.png">
    </center>
</div>

# ⛅ Cumulus Geographica – Cloud Native Geospatial i praksis (https://bit.ly/cumulus-geo) 🗺️

<a target="_blank" href="https://colab.research.google.com/github/kartAI/skygeo/blob/skygeo-workshop/src/skygeo-workshop/Cumulus_Geographica.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



---

Velkommen til denne workshopen! Her skal du lære å jobbe med geografiske data på en moderne og effektiv måte – selv når datasettene inneholder **mange milliarder datapunkter**.

**I workshopen vil du lære å:**
- Hente filtrerte geografiske data direkte fra skylagring uten å laste ned hele datasett
- Bruke DuckDB til å kjøre SQL mot store Parquet-filer
- Bruke GeoPandas til romlig analyse i Python
- Visualisere resultater interaktivt i notebooken

---

## ⛅ Hva er Cloud Native Geospatial (CNG)?

Tradisjonelt måtte man laste ned hele datafiler før man kunne analysere dem. Med Cloud Native Geospatial-teknologi kan vi i stedet lese _akkurat de delene av filen vi trenger_ direkte fra nettet – uten å laste ned resten.

To viktige konsepter:

**📦 GeoParquet**
Et filformat som lagrer geografiske data kolonnevis. Det gjør det mulig å lese kun utvalgte kolonner og rader – veldig effektivt for store datasett. Les mer: [geoparquet.org](https://geoparquet.org)

**🗂️ Hive-partitioning**
Dataene er organisert (partitioned) i mapper etter tema og type, f.eks. `theme=buildings/type=building/`. Med `*` som wildcard leser DuckDB automatisk alle relevante filer. Dette gjør det enkelt å filtrere på tvers av mange filer uten å tenke på filstrukturen.

**🔎 Predicate pushdown**
Når du filtrerer på f.eks. bbox eller kategori, sender DuckDB filteret ned til selve filen – og leser bare de delene som faktisk matcher. Resultatet: dramatisk raskere spørringer mot store datasett.

Vi bruker **Overture Maps** som eksempel – et åpent globalt geodatasett med milliarder av datapunkter og geometrier slik som bygninger, veier og steder (places) fra hele verden. Vi skal også bruke noen norske datasett for å lære forskjeller mellom globale datamengder og mindre lokale datasett. 

---
> 💡 **Tips til nybegynnere:** Du trenger ikke forstå alt med en gang. Kjør cellene i rekkefølge, les forklaringene, og eksperimenter gjerne med verdiene i koden!

## ⚙️ Steg 1: Installere pakker og sette opp miljøet

Før vi kan begynne må vi installere nødvendige Python-pakker. I Google Colab bruker vi `%pip install` for å gjøre dette direkte i notebooken.

**Hva installerer vi?**
| Pakke | Hva brukes det til? |
|---|---|
| `geopandas` | Lese, analysere og visualisere geografiske data |
| `duckdb` | Kjøre SQL-spørringer direkte mot store datafiler |
| `lonboard` | Rask, interaktiv kartvisualisering av store datasett |
| `folium` | Interaktive kart i notebooken |
| `fsspec` / `adlfs` | Lese filer fra Azure Blob Storage |

> ⏳ **Obs:** Installasjonen kan ta 1–2 minutter. Vent til cellen er ferdig (du ser et tall i `[ ]` til venstre) før du går videre.

In [2]:
%%capture

%pip install dotenv geopandas folium matplotlib mapclassify duckdb lonboard fsspec adlfs

import os
import geopandas as gpd
import duckdb
from shapely import wkb
from pathlib import Path
from lonboard import Map, PathLayer, viz
import folium

kommuner_path = "https://kartaistorage.blob.core.windows.net/skygeo/workshopdata/kommuner_n1000.parquet"
kirkebygg_path = "https://kartaistorage.blob.core.windows.net/skygeo/workshopdata/kirkebygg_forenklet.parquet"

os.makedirs('./tmp', exist_ok=True)

## 🦆 Steg 2: Sett opp DuckDB

**Hva er DuckDB?**
DuckDB er en rask, in-memory database som vi skal bruke direkte i Python – ingen server nødvendig. Den er spesielt god til å lese og analysere store datafiler (som Parquet) uten å laste hele datasettet inn i minnet. DuckDB kan kjøre som selvstendig klient og har bindinger til mange språk, fks Python - og også direkte i nettleser via WASM. 

Vi bruker DuckDB med `SPATIAL`-utvidelsen, som gir oss geografiske funksjoner som `ST_Intersects`, `ST_Buffer` og `ST_Transform`.

For å lese data fra Overture Maps (som er lagret på S3 i USA) setter vi også `s3_region`.

In [3]:
# setup duckdb connection
# Initialize DuckDB connection
conn = duckdb.connect(':memory:')

#install spatial
conn.execute("INSTALL spatial;")

# Load spatial extension
conn.execute("LOAD spatial;")

# Set S3 region for Overture Maps
conn.execute("SET s3_region='us-west-2';")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 🌍 Steg 3: Hent data fra Overture Maps

**Hva er Overture Maps?**
Overture Maps Foundation er et åpent initiativ som samler geografiske data fra hele verden – bygninger, veier, steder (places), og mer. Dataene er organisert som Parquet-filer lagret på S3 (Amazon sin skylagring). Se mer om [Overture Maps](https://docs.overturemaps.org/)

**Nøkkelkonsepter:**

- **Hive-partitioning** – Dataene er delt opp i mapper etter `theme` og `type`. Med `*` som wildcard leser DuckDB automatisk kun relevante filer.
- **Byte-range streaming** – Parquet-formatet lagrer en indeks (footer) på slutten av filen som beskriver hvor ulike kolonner og rader befinner seg. Det gjør det mulig å be om _nøyaktig de delene av filen vi trenger_ via HTTP byte-range requests – uten å laste ned hele filen.
- **Predicate pushdown** – Parquet-formatet lagrer også minimumsverdi og maksimumsverdi per kolonne i hver "row group". Når vi filtrerer på f.eks. `bbox`, kan DuckDB bruke denne statistikken til å hoppe over row groups som åpenbart ikke matcher – igjen uten å lese dem. Dette er en egenskap ved Parquet-formatet; DuckDB utnytter det.
- **bbox** – En bounding box (geografisk avgrensningsboks): `[min_lon, min_lat, max_lon, max_lat]`

**Oppgave:**
1. Kjør cellen under og se på resultatene. _Obs! Det kan ta fra 30 sekunder til noen minutter avhengig av bbox og netthastighet_
2. Prøv å **endre bbox** til et sted du kjenner – bruk [bboxfinder.com](https://bboxfinder.com/) for å finne koordinater.
3. Bytt ut kategorien (f.eks. `bakery`, `coffee_shop`, `pharmacy`) – se alle kategorier her: [Overture categories](https://github.com/OvertureMaps/schema/blob/main/docs/schema/concepts/by-theme/places/overture_categories.csv)

In [ ]:
bbox_kristiansand = [7.875366,58.081425,8.099899,58.206042]
bbox = bbox_kristiansand
# Query pizza restaurants and pizza_delivery_services in Kristiansand from Overture Maps

# Finn siste versjon av Overture Maps i S3-bucketen ved å spørre STAC-katalogen til Overture Maps. 
# STAC-katalogen inneholder metadata om alle versjoner av Overture Maps, inkludert den siste versjonen. Vi kan bruke en SQL-spørring for å hente ut den siste versjonen fra katalogen.
latest_sql = "SELECT latest FROM 'https://stac.overturemaps.org/catalog.json';"
latest_version = conn.sql(latest_sql).df().iloc[0, 0]

print(f"Latest version of Overture Maps: {latest_version}")

print("--" * 20)

query = f"""
SELECT
    id,
    names.primary as name,
    confidence,
    categories.primary as category,
    CAST(categories as JSON) as categories,
    CAST(socials AS JSON) as socials,
    geometry
FROM
    read_parquet('s3://overturemaps-us-west-2/release/{latest_version}/theme=places/type=place/*', 
                 filename=true, hive_partitioning=1)
WHERE
    categories.primary IN ('pizza_restaurant','pizza_delivery_service')
    AND bbox.xmin BETWEEN {bbox[0]} AND {bbox[2]}
    AND bbox.ymin BETWEEN {bbox[1]} AND {bbox[3]}
"""

print("Executing query")
#print sql
print(query)

print("--" * 20)

# Execute query
result = conn.sql(query)

display(result.df())


Latest version of Overture Maps: 2026-02-18.0
----------------------------------------
Executing query

SELECT
    id,
    names.primary as name,
    confidence,
    categories.primary as category,
    CAST(categories as JSON) as categories,
    CAST(socials AS JSON) as socials,
    geometry
FROM
    read_parquet('s3://overturemaps-us-west-2/release/2026-02-18.0/theme=places/type=place/*', 
                 filename=true, hive_partitioning=1)
WHERE
    categories.primary IN ('pizza_restaurant','pizza_delivery_service')
    AND bbox.xmin BETWEEN 7.875366 AND 8.099899
    AND bbox.ymin BETWEEN 58.081425 AND 58.206042

----------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id,name,confidence,category,categories,socials,geometry
0,bdc1c5bd-b924-448f-b72e-b903293bdeff,Happy Time Møvig,0.950063,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":[""re...","[""https://www.facebook.com/470935122760751""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
1,f3b176c9-1494-4552-a8e7-39515642f796,Pizza Show Vågsbygd,0.950063,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":[""re...","[""https://www.facebook.com/310729392892191""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
2,dfd50477-1f9d-44c8-8f90-0463f6675b7c,Domino's,0.400000,pizza_delivery_service,"{""primary"":""pizza_delivery_service"",""alternate...",None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
3,fdee4f58-a433-48bf-88b8-1245e2a0a607,Domino's Pizza,1.000000,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":null}",None,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
4,87b99033-b5ad-4d57-8263-0d0f74cdefa4,Pizzabakeren,0.970038,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":[""fa...","[""https://www.facebook.com/128311913852001""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
5,d40e5b20-7b6f-4e10-b169-ba01846fbb07,Every Day Pizza & Gyros,0.950063,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":[""re...","[""https://www.facebook.com/397846736749587""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
6,a209749f-350c-4077-8499-ad572d0005d6,Banda Santi,0.570194,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":null}","[""https://www.facebook.com/560995553771926""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
7,fe8fa4e8-1ff7-4958-bd5d-905d11d46823,Pizzabakeren Grim,0.970038,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":[""fa...","[""https://www.facebook.com/307286352784376""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
8,264df190-9378-43df-9479-e091e7132ba9,Kvadraturen grill & kiosk,0.639236,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":[""re...","[""https://www.facebook.com/471456249716446""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."
9,ea41d8dc-0f6e-44f1-92f7-85a5a641fa1b,Pizzeria,0.950063,pizza_restaurant,"{""primary"":""pizza_restaurant"",""alternate"":null}","[""https://www.facebook.com/338808489493076""]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


### 💾 Lagre spørringen som en tabell i DuckDB

I stedet for å kjøre samme spørring om og om igjen, kan vi lagre resultatet som en **tabell** i DuckDB. Det er raskere og gjør det enklere å jobbe videre med dataene.

`CREATE OR REPLACE TABLE` betyr: opprett tabellen, eller erstatt den hvis den allerede finnes. Vi bruker spørringen ovenfor som "grunnlag" for å lage den nye tabellen.

In [ ]:

conn.sql(f"""
CREATE OR REPLACE TABLE overturemap_results AS
    {query}
""")

conn.sql("SELECT * FROM overturemap_results").df().head(5)

In [ ]:
# Verify the geometry column exists and its type
r = conn.sql("DESCRIBE overturemap_results")
r.df()

### 📂 Lagre resultatet som en Parquet-fil på disk

**Hva er Parquet?**
Parquet er et kolonnebasert filformat som er mye brukt for geografiske og analytiske data. Det er svært effektivt å lagre og lese, og støtter metadata for geometri (GeoParquet).

Her eksporterer vi DuckDB-resultatet til en Parquet-fil, slik at vi kan lese det videre med blant annet GeoPandas.

In [ ]:
# export the duckdb result to a parquet file and read it with geopandas
tmp_parquet_path = "./tmp/duckdb_result.parquet"
result.to_parquet(tmp_parquet_path)

### 🗺️ Lag et interaktivt kart av resultatet

Vi bruker **[lonboard](https://developmentseed.org/lonboard/)** for rask, interaktiv visualisering. Lonboard er bygget på WebGL og håndterer store mengder punkter og geometrier som vanlige kartbiblioteker sliter med.

`viz()` er en snarvei som automatisk velger riktig kartlag basert på geometritypen i dataene. Vi kan gi inn DuckDB-resultater direkte i `viz()`. 

In [ ]:
map = viz(result)

### 🐼 Bruke GeoPandas på resultatet

**Hva er GeoPandas?**
[GeoPandas](https://geopandas.org/) er Python-biblioteket for geografisk dataanalyse. Det utvider [Pandas](https://pandas.pydata.org/) (tabelldata) med støtte for geometri og kartprojeksjoner. Data lastes inn i en `GeoDataFrame` – en tabell der én kolonne inneholder geometrier (punkter, linjer, polygoner).

Vi leser Parquet-filen vi nettopp lagret enkelt med `read_parquet()` og visualiserer den på to måter:
- `.plot()` – rask, statisk visualisering (matplotlib)
- `.explore()` – interaktivt kart i notebooken (folium)

In [ ]:
# read the parquet file with geopandas
gdf = gpd.read_parquet(tmp_parquet_path)
gdf.head()

In [ ]:
gdf.plot()
gdf.explore()

## 📐 Del 2: Romlig analyse med GeoPandas

Nå skal vi gjøre en enkel, men reell geografisk analyse: **Finn alle kirker i og nær en gitt kommune.**

Vi bruker to datasett:
- **Kommunegrenser** – polygoner for alle kommuner i Norge (N1000-skala)
- **Kirkebygg** – punktdata med navn, kapasitet og byggemateriale

Dataene er lagret som GeoParquet-filer på Azure Blob Storage og leses direkte med GeoPandas.

In [ ]:
kommune_uri = "az://skygeo/workshopdata/kommuner_n1000.parquet"
account_name = "kartaistorage"

kommune_original = gpd.read_parquet(
    kommune_uri, 
    storage_options={"account_name": account_name}
)

kirkebygg_uri = "az://skygeo/workshopdata/kirkebygg_forenklet.parquet"

kirkebygg = gpd.read_parquet(
    kirkebygg_uri, 
    storage_options={"account_name": account_name}
)

### 🔵 Filtrer ut én kommune og lag en buffersone

Vi jobber med **Lillehammer kommune** (`kommunenummer = 3305`).

**Hva er en buffer?**
En buffer er en sone rundt en geometri med en gitt avstand – her 15 km. Vi bruker bufferen for å finne kirker som ligger _nær_ kommunen, ikke bare inni den.

**Merk koordinatsystemet:**
- Geografiske koordinater (lengde-/breddegrad, EPSG:4326) er ikke egnet for avstandsmåling i meter.
- Vi transformerer til **EPSG:3857** (metrisk projeksjoner) for å lage en 15 km buffer, og konverterer tilbake til EPSG:4326 etterpå.

In [ ]:
kommune_original = kommune_original[kommune_original["kommunenummer"] == "3305"][["kommunenummer", "kommunenavn", "geometry"]].copy()

kommune_buffer = kommune_original.copy()

kommune_buffer = kommune_buffer.to_crs(3857) # REFLEKSJON: Hvorfor gjør vi dette? Er det riktig?

kommune_buffer["geometry"] = kommune_buffer.buffer(15_000) # REFLEKSJON: Hva gjør vi her?

kommune_buffer = kommune_buffer.to_crs(4326)

### ✂️ Klipp kirker innen kommune og buffer

`.clip()` i GeoPandas klipper et datasett til å kun inneholde geometrier som er innenfor en gitt polygon.

Vi gjør dette to ganger:
1. Klipp kirker til **kommunegrensen** (nøyaktig grense)
2. Klipp kirker til **buffersonen** (15 km utenfor grensen)

Deretter teller vi antall kirker og summerer kapasitet.

In [ ]:
kirker_i_original = kirkebygg[["bygningsnavn", "godkjentkapasitet", "hovedmateriale", "geometry"]].clip(kommune_original)

kirker_i_original_antall = len(kirker_i_original)

kirker_i_original_kapasitet = kirker_i_original["godkjentkapasitet"].sum()

display(f"Antall kirker i kommune er {kirker_i_original_antall} og rommer {kirker_i_original_kapasitet} personer.")

kirker_i_buffer = kirkebygg[["bygningsnavn", "godkjentkapasitet", "hovedmateriale", "geometry"]].clip(kommune_buffer)

kirker_i_buffer_antall = len(kirker_i_buffer)

kirker_i_buffer_kapasitet = kirker_i_buffer["godkjentkapasitet"].sum()

display(f"Antall kirker i buffer er {kirker_i_buffer_antall} og rommer {kirker_i_buffer_kapasitet} personer.")

### 🗺️ Visualiser resultatene med Folium

Vi lager et interaktivt kart med [Folium](https://folium.readthedocs.io/en/latest/) som viser:
- **Blå linje** – kommunegrensen
- **Rød stiplet linje** – buffersonen (15 km utenfor)
- **Svarte prikker** – kirkebygg (trykk på en prikk for å se bygningsnavn)

In [ ]:
xmin, ymin, xmax, ymax = kommune_buffer.total_bounds.tolist()

m = folium.Map(
    tiles="cartodb positron",
)

m.fit_bounds([[ymin, xmin], [ymax, xmax]])

folium.GeoJson(
    data=kommune_original,
    color="blue",
    fill_opacity=0.0,
    weight=2.0,
).add_to(m)

folium.GeoJson(
    data=kommune_buffer,
    color="red",
    fill_opacity=0.0,
    weight=1.0,
    dash_array=[10.0, 5.0],
).add_to(m)

folium.GeoJson(
    data=kirker_i_buffer,
    tooltip=folium.GeoJsonTooltip(fields=["bygningsnavn"]),
    marker=folium.CircleMarker(
        color="black",
        fill_color="white",
        fill_opacity=1.0,
        radius=5.0,
        weight=1,
    ),
).add_to(m)

m

## 🚀 Del 3: Analyser på store datasett med DuckDB

GeoPandas er flott når dataene passer i RAM (typisk noen hundre MB). Men hva når datasettet er på **mange gigabyte eller terabyte**?

Da må vi tenke annerledes. DuckDB løser dette ved å:
1. **Lese kun nødvendige kolonner og rader** fra filen (predicate pushdown)
2. **Filtrere tidlig**, slik at minst mulig data lastes inn i minnet
3. **Utføre romlige operasjoner** direkte i SQL

I de neste cellene bruker vi DuckDB til å:
- Hente kommunegeometri for Ringerike
- Laste ned kun bygninger innenfor bounding box fra Overture Maps og lagre
- Gjøre nøyaktig romlig intersect på den lokalt lagrede filen

### To-stegs strategi: bbox-filter → nøyaktig intersect

Nedlasting av alle bygninger globalt er umulig – det er hundrevis av gigabyte. Strategien er:

**Steg 1 – Grov filtrering (bbox):** Last ned kun bygninger som overlapper med kommunens bounding box. Dette er raskt fordi Overture Maps-filene har `bbox`-kolonner DuckDB kan pushdown-filtrere på uten å lese hele filen.

**Steg 2 – Presis filtrering (spatial intersect):** Kjør `ST_Intersects` på den lille, lokalt lagrede filen for å fjerne bygg som er i bbox men utenfor kommunegrensen.

> 💡 Steg 1 er en "grov sil" – vi laster det meste vi trenger. Steg 2 er den presise silen. Aldri gjør romlig intersect direkte mot remote-filer med milliarder rader!

In [ ]:
# Kommune 3305: transformer til EPSG:25833, buffer 15000 meter, transformer tilbake for intersect

conn.sql(f"""
    CREATE OR REPLACE TABLE kirkebygg_i_kommune_3305 AS
    WITH kommune_buffer AS (
        SELECT
            ST_Transform(
                ST_Buffer(
                    ST_Transform(geometry, 'EPSG:4326', 'EPSG:25833'),
                    15000
                ),
                'EPSG:25833',
                'EPSG:4326'
            ) AS geometry
        FROM read_parquet('{kommuner_path}')
        WHERE CAST(kommunenummer AS VARCHAR) = '3305'
        LIMIT 1
    )
    SELECT k.*
    FROM read_parquet('{kirkebygg_path}') AS k
    JOIN kommune_buffer AS b
      ON ST_Intersects(k.geometry, b.geometry)
""")

# lagre resultater som parquet
conn.sql("COPY kirkebygg_i_kommune_3305 TO './tmp/kirkebygg_i_kommune_3305.parquet' (FORMAT PARQUET)")

conn.sql("SELECT COUNT(*) AS antall_kirkebygg FROM kirkebygg_i_kommune_3305").df()

In [ ]:
# Stegvis for å tvinge tidlig filtrering (predicate pushdown)
os.makedirs('./tmp', exist_ok=True)

# 1) Hent kommune 3305-geometri som egen tabell
conn.sql(f"""
    CREATE OR REPLACE TABLE kommune_3305 AS
    SELECT geometry
    FROM read_parquet('{kommuner_path}')
    WHERE CAST(kommunenummer AS VARCHAR) = '3305'
    LIMIT 1
""")

print("Kommune 3305 geometri lastet inn i DuckDB.")

# 2) Hent bbox-verdier til Python (literals i neste query)
xmin, ymin, xmax, ymax = conn.sql("""
    SELECT
        ST_XMin(geometry) AS xmin,
        ST_YMin(geometry) AS ymin,
        ST_XMax(geometry) AS xmax,
        ST_YMax(geometry) AS ymax
    FROM kommune_3305
""").fetchone()

print(f"BBOX for kommune 3305: xmin={xmin}, ymin={ymin}, xmax={xmax}, ymax={ymax}")

print("Kjører spørring mot Overture Maps ..........")

# 3) Kun bbox-filter mot Overture (remote), og lagre lokalt parquet
conn.sql(f"""
    COPY (
        SELECT
            id,
            CAST(sources AS JSON) AS sources,
            subtype,
            class,
            geometry
        FROM read_parquet(
            's3://overturemaps-us-west-2/release/2026-02-18.0/theme=buildings/type=building/*',
            filename=true,
            hive_partitioning=1
        )
        WHERE
            bbox.xmax >= {xmin}
            AND bbox.xmin <= {xmax}
            AND bbox.ymax >= {ymin}
            AND bbox.ymin <= {ymax}
    ) TO './tmp/overture_buildings_bbox_3305.parquet' (FORMAT PARQUET)
""")

print("Bygningsdata for bbox lastet ned og lagret lokalt.")

# 4) Presis spatial intersect på den lokale, forhåndsfiltrerte filen
conn.sql("""
    CREATE OR REPLACE TABLE overture_buildings_i_kommune_3305 AS
    SELECT b.*
    FROM read_parquet('./tmp/overture_buildings_bbox_3305.parquet') b
    CROSS JOIN kommune_3305 k
    WHERE ST_Intersects(b.geometry, k.geometry)
""")

print("Presis spatial intersect utført på lokal fil.")

conn.sql("SELECT COUNT(*) AS antall_bygninger FROM overture_buildings_i_kommune_3305").df()

In [ ]:
conn.sql("SELECT * FROM overture_buildings_i_kommune_3305").df()

### 🏢 Finn store bygninger som ikke er boliger i kommunen

Nå har vi alle bygninger i kommunen lastet inn. La oss filtrere ut de som er interessante:
- **Ikke boligbygg** (`subtype NOT IN ('residential')`)
- **Over 10 000 m²** – store nærings- eller offentlige bygg

`ST_Area` beregner arealet, men vi må transformere til et metrisk koordinatsystem (EPSG:25833) for å få arealet i kvadratmeter.

In [ ]:
res = conn.sql("""
    SELECT 
        *,
        st_area(st_transform(geometry,'EPSG:4326','EPSG:25833')) as area,
        st_astext(geometry) geometry_wkt
    FROM overture_buildings_i_kommune_3305
    WHERE
        subtype NOT IN ('residential')
        AND area > 10000
         """)
display(res.df())

viz(res)

## 💾 Eksportere resultater

Når du har gjort analysene dine kan du eksportere resultatene til ulike formater:

| Format | Bruksområde |
|---|---|
| **GeoParquet** | Videre analyse i Python/DuckDB, deling med andre |
| **GeoJSON** | Åpne i QGIS, bruke i webkart |
| **CSV** | Åpne i Excel (uten geometri) |
| **HTML-kart** | Del interaktivt kart i nettleser |

Kjør cellen under for å eksportere resultatene dine.

In [ ]:
export_dir = "./tmp/export"
os.makedirs(export_dir, exist_ok=True)

# --- 1. Lagre DuckDB-resultatet som GeoParquet ---
res_parquet_path = f"{export_dir}/eksport.parquet"
conn.sql(f"""
    COPY 
        (SELECT * FROM kirkebygg_i_kommune_3305) 
    TO '{res_parquet_path}' (FORMAT PARQUET)
""")
print(f"✅ GeoParquet lagret: {res_parquet_path}")

# Les inn som GeoDataFrame for videre eksport
gdf_export = gpd.read_parquet(res_parquet_path)

# --- 2. Eksporter til GeoJSON (kan åpnes i QGIS og webkart) ---
geojson_path = f"{export_dir}/eksport.geojson"
gdf_export.to_file(geojson_path, driver="GeoJSON")
print(f"✅ GeoJSON lagret: {geojson_path}")

# --- 3. Eksporter til CSV (uten geometri, kan åpnes i Excel) ---
csv_path = f"{export_dir}/eksport.csv"
gdf_export.drop(columns="geometry").to_csv(csv_path, index=False)
print(f"✅ CSV lagret: {csv_path}")

# --- 4. Lag og lagre et interaktivt HTML-kart ---
xmin, ymin, xmax, ymax = kommune_buffer.total_bounds.tolist()
kart = folium.Map(tiles="cartodb positron")
kart.fit_bounds([[ymin, xmin], [ymax, xmax]])

# Kommunegrense (blå solid linje)
folium.GeoJson(
    data=kommune_original,
    name="Kommunegrense",
    style_function=lambda f: {"color": "blue", "fillOpacity": 0.0, "weight": 2},
).add_to(kart)

# Buffersone (rød stiplet linje)
folium.GeoJson(
    data=kommune_buffer,
    name="Buffersone (15 km)",
    style_function=lambda f: {"color": "red", "fillOpacity": 0.0, "weight": 1, "dashArray": "10 5"},
).add_to(kart)

# Kirkebygg (punkter med tooltip)
folium.GeoJson(
    data=gdf_export,
    name="Kirkebygg",
    tooltip=folium.GeoJsonTooltip(fields=["bygningsnavn", "godkjentkapasitet", "hovedmateriale"]),
    marker=folium.CircleMarker(
        color="black",
        fill_color="white",
        fill_opacity=1.0,
        radius=5.0,
        weight=1,
    ),
).add_to(kart)

folium.LayerControl().add_to(kart)

html_path = f"{export_dir}/eksport_kart.html"
kart.save(html_path)
print(f"✅ HTML-kart lagret: {html_path}")

kart

: 

## 🎯 Oppgave: Finn alle Places i kommunen!

Bruk det du har lært til å finne alle steder (Places) fra Overture Maps innenfor Ringsaker kommune.

**Tips:**
- Bruk bbox-koordinatene vi hentet i steg 2 (`xmin`, `ymin`, `xmax`, `ymax`)
- Overture Maps Places ligger på: `s3://overturemaps-us-west-2/release/2026-02-18.0/theme=places/type=place/*`
- Filtrer på `bbox.xmin`, `bbox.xmax`, `bbox.ymin`, `bbox.ymax`

---

### ⭐ Ekstraoppgave: Finn alle Places innenfor 100 meter fra en kirke

Kombiner kirkedataene med Places-dataene for å finne hvilke steder (kafeer, butikker, osv.) som ligger i nærheten av kirker.

**Tips:** Bruk `ST_DWithin(geom_a, geom_b, distance_meters)` eller `ST_Buffer` + `ST_Intersects`.